# 🎯 Distributed Face Detection Service for Google Colab

This notebook sets up a face detection microservice that:
1. Receives frames from your local PC via HTTP POST
2. Runs face detection and recognition using InsightFace
3. Optionally forwards results to your cloud backend
4. Returns detection results to your PC for display

**Prerequisites:**
- Your local PC running `pc_frame_sender.py`
- Your cloud backend (Render) accessible via HTTPS
- ngrok or similar tunneling service installed on Colab

**Setup Instructions:**
1. Run all cells in this notebook
2. Start the tunnel (ngrok will be started automatically in cell 3)
3. Note the public URL provided by ngrok
4. Configure your PC's `pc_frame_sender.py` with this URL
5. Start your local PC frame sender
6. View results on your local PC preview window and/or cloud dashboard


In [ ]:
# Install required packages
!pip install insightface onnxruntime opencv-python fastapi uvicorn ngrok-py -q

In [ ]:
# Import required modules
import os
import sys
import subprocess
import time
import threading
from pathlib import Path
import numpy as np
import cv2
from IPython.display import clear_output, HTML, display
import json
import requests
from fastapi import FastAPI, File, UploadFile, HTTPException, Header
from fastapi.responses import JSONResponse
import uvicorn
import insightface
from insightface.app import FaceAnalysis
import ngrok


## 🔐 Configuration

Set your configuration values below. These will be used as environment variables for the detection service.

**Important:** You'll need to get these values from your existing setup:
- `CLOUD_BACKEND_URL`: Your Render backend URL (e.g., https://your-service.onrender.com)
- `CLOUD_API_KEY`: The edge API key from your backend's environment
- `COLAB_API_KEY`: A separate API key for securing the Colab service (generate a random string)
- `KNOWN_FACES_DIR`: Path where known faces will be stored (we'll create this)
- `GALLERY_PATH`: Path where gallery.npz will be stored/computed
- `SIMILARITY_THRESHOLD`: Should match your local config (0.35)
- `USE_GPU`: Set to "true" if you want to use GPU (requires Colab Pro or T4 runtime)


In [ ]:
# ===== CONFIGURE THESE VALUES =====
# Get these from your existing facial recognition setup

# Your Render backend URL
CLOUD_BACKEND_URL = "https://your-service.onrender.com"  # <-- CHANGE THIS

# Edge API key from your backend (same as what your local edge uses)
# You can find this in your backend's environment or .env file
CLOUD_API_KEY = "your-edge-api-key-here"  # <-- CHANGE THIS

# Separate API key for securing the Colab service (generate a random string)
COLAB_API_KEY = "your-colab-service-api-key"  # <-- CHANGE THIS

# Detection threshold (should match your local facial_recognition/config.yaml)
SIMILARITY_THRESHOLD = 0.35

# Set to "true" to use GPU (requires Colab Pro or T4 runtime)
USE_GPU = "false"

# ===== END CONFIGURATION =====

# Set environment variables
os.environ['CLOUD_BACKEND_URL'] = CLOUD_BACKEND_URL
os.environ['CLOUD_API_KEY'] = CLOUD_API_KEY
os.environ['COLAB_API_KEY'] = COLAB_API_KEY
os.environ['SIMILARITY_THRESHOLD'] = SIMILARITY_THRESHOLD
os.environ['USE_GPU'] = USE_GPU

print("Configuration set:")
print(f"  Cloud Backend URL: {CLOUD_BACKEND_URL}")
print(f"  Cloud API Key: {'*' * len(CLOUD_API_KEY) if CLOUD_API_KEY else 'Not set'}")
print(f"  Colab Service API Key: {'*' * len(COLAB_API_KEY) if COLAB_API_KEY else 'Not set'}")
print(f"  Similarity Threshold: {SIMILARITY_THRESHOLD}")
print(f"  Use GPU: {USE_GPU}")


## 📁 Setup Known Faces Directory

We'll create a directory structure for known faces and optionally upload your existing known faces.

**Option 1:** Use your existing known_faces folder (if you have it locally)
**Option 2:** Start with empty directory and add faces later via the dashboard

For Option 1, you would upload your known_faces folder using the file upload below.
For Option 2, we'll just create the directory structure.

In [ ]:
# Create known faces directory structure
KNOWN_FACES_DIR = os.getenv("KNOWN_FACES_DIR", "known_faces")
GALLERY_PATH = os.getenv("GALLERY_PATH", "known_faces/gallery.npz")

# Create directory if it doesn't exist
Path(KNOWN_FACES_DIR).mkdir(parents=True, exist_ok=True)

print(f"Known faces directory: {KNOWN_FACES_DIR}")
print(f"Gallery path: {GALLERY_PATH}")

# List any existing content
if os.path.exists(KNOWN_FACES_DIR):
    items = os.listdir(KNOWN_FACES_DIR)
    print(f"Items in known_faces directory: {items}")
    if items:
        for item in items:
            item_path = os.path.join(KNOWN_FACES_DIR, item)
            if os.path.isdir(item_path):
                img_count = len([f for f in os.listdir(item_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
                print(f"  📁 {item}: {img_count} images")
            else:
                print(f"  📄 {item}")
else:
    print("Known faces directory is empty")


## 📤 Upload Your Known Faces (Optional)

If you have an existing known_faces folder from your local setup, upload it here.

**To upload:**
1. Click the file upload button below
2. Select your known_faces folder (zip it first if it's a folder)
3. Wait for upload to complete
4. Run the next cell to extract if needed

If you don't have existing known faces or want to start fresh, skip this step.

In [ ]:
# Uncomment the following line to upload files
# from google.colab import files
# uploaded = files.upload()

# If you uploaded a zip file, extract it
# import zipfile
# for filename in uploaded.keys():
#     if filename.endswith('.zip'):
#         with zipfile.ZipFile(filename, 'r') as zip_ref:
#             zip_ref.extractall(KNOWN_FACES_DIR)
#         print(f"Extracted {filename} to {KNOWN_FACES_DIR}")

# Show updated directory structure
if os.path.exists(KNOWN_FACES_DIR):
    items = os.listdir(KNOWN_FACES_DIR)
    print(f"\nUpdated known_faces directory:")
    for item in items:
        item_path = os.path.join(KNOWN_FACES_DIR, item)
        if os.path.isdir(item_path):
            img_count = len([f for f in os.listdir(item_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            print(f"  📁 {item}: {img_count} images")
        else:
            print(f"  📄 {item}")


## 🚀 Start the Detection Service

This cell will:
1. Initialize the InsightFace model
2. Load or compute the known faces gallery
3. Start the FastAPI server
4. Start the ngrok tunnel
5. Display the public URL for your PC to connect to

⚠️ **Note:** This cell will run indefinitely until you stop it.
   Do not run other cells while this is active.
   To stop, click the interrupt button (■) or press Ctrl+M twice.


In [ ]:
# Global variables for the detection service
detector_app = None
known_gallery_embeddings = None
known_gallery_labels = None
similarity_threshold = float(os.getenv("SIMILARITY_THRESHOLD", "0.35"))
cloud_backend_url = os.getenv("CLOUD_BACKEND_URL")
cloud_api_key = os.getenv("CLOUD_API_KEY")

print("Initializing detection service...")
print(f"Similarity threshold: {similarity_threshold}")
print(f"Cloud backend URL: {cloud_backend_url}")
print(f"Cloud API key configured: {bool(cloud_api_key)}")


In [ ]:
# Detection service implementation (same as colab_detection_service.py but adapted for notebook)
from fastapi import FastAPI, File, UploadFile, HTTPException, Header
from fastapi.responses import JSONResponse
import uvicorn
import uuid
from datetime import datetime, timezone
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="Remote Face Detection Service")

def create_detection_template() -> dict:
    return {
        "event_id": str(uuid.uuid4()),
        "embedding": [],
        "device_id": "local_pc",
        "sequence_number": 0,
        "camera_id": "webcam",
        "profile_id": None,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "status": "unknown",
        "confidence": 0.0,
        "bbox": [0, 0, 0, 0],
        "liveness_score": 0.0,
        "age": 0,
        "gender": "unknown",
        "wearing_mask": False,
        "wearing_glasses": False,
        "priority": "normal",
        "config_version": 1,
        "detection_model_version": "scrfd_500m_bnkps_v1",
        "embedding_model_version": "w600k_mbf_v1",
        "gallery_version": 1,
        "threshold_version": 1,
        "camera_config_version": 1,
        "algorithm_version": "temporal_fusion_v2",
        "version_bundle_hash": ""
    }

@app.on_event("startup")
async def startup_event():
    global detector_app, known_gallery_embeddings, known_gallery_labels
    logger.info("Starting up face detection service...")

    # Initialize InsightFace detector
    logger.info("Loading InsightFace model...")
    try:
        ctx_id = 0 if os.getenv("USE_GPU", "false").lower() == "true" else -1
        detector_app = FaceAnalysis(name='buffalo_s', 
                                  providers=['CUDAExecutionProvider'] if ctx_id == 0 else ['CPUExecutionProvider'])
        detector_app.prepare(ctx_id=ctx_id, det_size=(640, 640))
        logger.info(f"InsightFace model loaded successfully (ctx_id: {ctx_id})")
    except Exception as e:
        logger.error(f"Failed to load InsightFace model: {e}")
        raise

    # Load known faces gallery
    logger.info("Loading known faces gallery...")
    gallery_path = os.getenv("GALLERY_PATH", "known_faces/gallery.npz")
    known_faces_dir = os.getenv("KNOWN_FACES_DIR", "known_faces")

    try:
        # Try to load precomputed gallery first
        if os.path.exists(gallery_path):
            logger.info(f"Loading gallery from {gallery_path}")
            data = np.load(gallery_path, allow_pickle=True)
            known_gallery_labels = list(data['labels'])
            known_gallery_embeddings = np.asarray(data['embeddings'])
            logger.info(f"Loaded gallery with {len(known_gallery_labels)} identities")
        else:
            # Compute gallery from known_faces directory
            logger.info(f"Computing gallery from directory: {known_faces_dir}")
            known_gallery_embeddings, known_gallery_labels = compute_gallery_from_directory(known_faces_dir)
            # Optionally save the computed gallery
            if known_gallery_embeddings is not None and len(known_gallery_embeddings) > 0:
                save_path = Path(gallery_path)
                save_path.parent.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(save_path,
                                  labels=np.array(known_gallery_labels, dtype=object),
                                  embeddings=known_gallery_embeddings)
                logger.info(f"Saved computed gallery to {gallery_path}")
    except Exception as e:
        logger.error(f"Failed to load known faces gallery: {e}")
        # Continue with empty gallery - will treat all faces as unknown
        known_gallery_embeddings = np.zeros((0, 512), dtype=np.float32)
        known_gallery_labels = []

def compute_gallery_from_directory(known_faces_dir: str):
    """Compute face embeddings from images in known_faces directory structure."""
    embeddings = []
    labels = []

    known_faces_path = Path(known_faces_dir)
    if not known_faces_path.exists():
        logger.warning(f"Known faces directory not found: {known_faces_dir}")
        return np.zeros((0, 512), dtype=np.float32), []

    logger.info(f"Scanning {known_faces_dir} for known faces...")
    for person_dir in known_faces_path.iterdir():
        if not person_dir.is_dir():
            continue

        person_name = person_dir.name
        person_embeddings = []

        # Process all images in the person's directory
        for img_path in person_dir.glob("*.[jp][pn]g"):  # jpg, jpeg, png
            try:
                img = cv2.imread(str(img_path))
                if img is None:
                    logger.warning(f"Could not read image: {img_path}")
                    continue

                # Detect faces
                faces = detector_app.get(img)
                if not faces:
                    logger.warning(f"No face detected in: {img_path}")
                    continue

                # Use the largest face (by bounding box area)
                largest_face = max(faces, key=lambda f:
                                 (f['bbox'][2] - f['bbox'][0]) * (f['bbox'][3] - f['bbox'][1]))
                embedding = largest_face.embedding

                if embedding is not None and len(embedding) == 512:
                    person_embeddings.append(embedding)
                    logger.debug(f"Added embedding for {person_name} from {img_path.name}")
                else:
                    logger.warning(f"Invalid embedding for {img_path}")
            except Exception as e:
                logger.error(f"Error processing {img_path}: {e}")

        if person_embeddings:
            # Average embeddings for this person
            avg_embedding = np.mean(person_embeddings, axis=0)
            # Normalize
            avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)
            embeddings.append(avg_embedding)
            labels.append(person_name)
            logger.info(f"Added {person_name} with {len(person_embeddings)} images")
        else:
            logger.warning(f"No valid embeddings found for {person_name}")

    if len(embeddings) == 0:
        logger.warning("No known faces found - gallery will be empty")
        return np.zeros((0, 512), dtype=np.float32), []

    return np.asarray(embeddings, dtype=np.float32), labels

def recognize_face(embedding: np.ndarray):
    """Recognize a face by comparing against the known gallery."""
    if known_gallery_embeddings is None or len(known_gallery_embeddings) == 0:
        return None, 0.0

    # Normalize the query embedding
    query_norm = embedding / np.linalg.norm(embedding)

    # Compute cosine similarity with all known embeddings
    similarities = np.dot(known_gallery_embeddings, query_norm)

    # Find the best match
    best_idx = np.argmax(similarities)
    best_similarity = similarities[best_idx]

    if best_similarity >= similarity_threshold:
        return known_gallery_labels[best_idx], float(best_similarity)
    else:
        return None, float(best_similarity)

@app.post("/detect")
async def detect_faces(
    file: UploadFile = File(...),
    x_api_key: Optional[str] = Header(None)
):
    global detector_app, known_gallery_embeddings, known_gallery_labels
    global similarity_threshold, cloud_backend_url, cloud_api_key

    # Optional API key validation
    expected_api_key = os.getenv("COLAB_API_KEY")
    if expected_api_key and x_api_key != expected_api_key:
        raise HTTPException(status_code=403, detail="Invalid API key")

    if detector_app is None:
        raise HTTPException(status_code=503, detail="Detection service not initialized")

    try:
        # Read and decode the image
        contents = await file.read()
        nparr = np.frombuffer(contents, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if img is None:
            raise HTTPException(status_code=400, detail="Invalid image format")

        # Run face detection
        faces = detector_app.get(img)

        detections = []
        sequence_offset = 0

        for i, face in enumerate(faces):
            # Extract bounding box and embedding
            bbox = face['bbox'].astype(int)  # [x0, y0, x1, y1]
            embedding = face.embedding

            if embedding is None or len(embedding) != 512:
                logger.warning(f"Skipping face {i} due to invalid embedding")
                continue

            # Recognize the face
            identity, confidence = recognize_face(embedding)

            # Create detection record
            detection = create_detection_template()
            detection["event_id"] = str(uuid.uuid4())
            detection["embedding"] = embedding.tolist()
            detection["device_id"] = "local_pc"
            detection["sequence_number"] = sequence_offset + i
            detection["camera_id"] = "webcam"
            detection["timestamp"] = datetime.now(timezone.utc).isoformat()
            detection["confidence"] = confidence
            detection["bbox"] = [int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])]

            if identity is not None:
                # Recognized face
                detection["identity"] = identity
                detection["status"] = "recognized"
            else:
                # Unknown face
                detection["identity"] = "Unknown"
                detection["status"] = "unknown"
                # Generate a temporary label for unknown faces
                detection["identity"] = f"Person {hash(str(embedding.tobytes())) % 10000}"

            # Extract additional attributes if available from the model
            if hasattr(face, 'age') and face.age is not None:
                detection["age"] = int(face.age)
            if hasattr(face, 'gender') and face.gender is not None:
                detection["gender"] = "male" if face.gender == 1 else "female"

            detections.append(detection)

        # Optionally forward detections to cloud backend
        if cloud_backend_url and cloud_api_key and detections:
            forwarded_count = 0
            for detection in detections:
                try:
                    # Prepare headers for cloud backend
                    headers = {
                        "X-API-Key": cloud_api_key,
                        "Content-Type": "application/json"
                    }

                    # Send to cloud backend
                    response = requests.post(
                        f"{cloud_backend_url}/api/detections",
                        json=detection,
                        timeout=5
                    )

                    if response.status_code == 200:
                        forwarded_count += 1
                    else:
                        logger.warning(f"Failed to forward detection to cloud: {response.status_code}")
                except Exception as e:
                    logger.error(f"Error forwarding detection to cloud: {e}")

            logger.info(f"Forwarded {forwarded_count}/{len(detections)} detections to cloud backend")

        # Return detection results
        return JSONResponse(content={
            "detections": detections,
            "processed_at": datetime.now(timezone.utc).isoformat(),
            "frame_shape": list(img.shape) if img is not None else [0, 0, 0]
        })

    except Exception as e:
        logger.error(f"Error in detect_faces: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health_check():
    return {
        "status": "healthy" if detector_app is not None else "initializing",
        "model_loaded": detector_app is not None,
        "gallery_size": len(known_gallery_labels) if known_gallery_labels else 0,
        "similarity_threshold": similarity_threshold,
        "cloud_backend_configured": cloud_backend_url is not None
    }

# Function to run the server
def run_server():
    port = 7860
    logger.info(f"Starting FastAPI server on port {port}")
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")


## 🔗 Start Ngrok Tunnel

This cell will start an ngrok tunnel to make your Colab service accessible from your local PC.

**If you don't have ngrok authtoken configured:**
1. Sign up at https://ngrok.com/
2. Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
3. Run: `ngrok config add-authtoken YOUR_TOKEN` in a terminal
4. Or uncomment and run the line below with your token

The tunnel will forward to your local port 7860 where the FastAPI server is running.

In [ ]:
# Start ngrok tunnel
# Uncomment and set your authtoken if needed
# ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN_HERE")

# Start tunnel
print("Starting ngrok tunnel...")
try:
    # Kill any existing tunnels
    ngrok.kill()
    
    # Start new tunnel
    tunnel = ngrok.connect(7860, "http")
    public_url = tunnel.public_url
    
    print("\n" + "="*50)
    print("🚀 NGROK TUNNEL ESTABLISHED")
    print("="*50)
    print(f"Public URL: {public_url}")
    print(f"Local port: 7860")
    print()
    print("📋 NEXT STEPS:")
    print(f"1. Configure your PC's pc_frame_sender.py with:")
    print(f"   REMOTE_URL = '{public_url}'")
    print(f"2. Make sure to use the same API key:")
    print(f"   API_KEY = '{COLAB_API_KEY}'")
    print()
    print("💡 TIP: Keep this notebook running while using the service.")
    print("   To stop, interrupt this cell (■) or close the notebook.")
    print("="*50)

except Exception as e:
    print(f"Error starting ngrok tunnel: {e}")
    print("Make sure you have ngrok installed and configured.")


## ▶️ Start Detection Service Server

Now start the FastAPI server in the background. This will keep running while you use the tunnel.

**Important:** Run this cell AFTER starting the ngrok tunnel above.

The server will:
- Listen for frame POST requests at http://localhost:7860/detect
- Process frames using InsightFace
- Return detection results
- Optionally forward to your cloud backend

💡 **To stop the service:** Interrupt this cell (■) or restart the runtime.

In [ ]:
# Start the FastAPI server in a background thread
import threading

def start_server_thread():
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    return server_thread

print("Starting detection service server...")
server_thread = start_server_thread()

# Give it a moment to start
time.sleep(3)

# Test the health endpoint
try:
    import requests
    health_response = requests.get("http://localhost:7860/health", timeout=5)
    if health_response.status_code == 200:
        health_data = health_response.json()
        print("✅ Detection service is healthy!")
        print(f"   Status: {health_data['status']}")
        print(f"   Model loaded: {health_data['model_loaded']}")
        print(f"   Gallery size: {health_data['gallery_size']} identities")
        print(f"   Similarity threshold: {health_data['similarity_threshold']}")
        print(f"   Cloud backend configured: {health_data['cloud_backend_configured']}")
    else:
        print(f"❌ Health check failed: {health_response.status_code}")
        print(f"   Response: {health_response.text}")
except Exception as e:
    print(f"❌ Could not connect to service: {e}")
    print("Make sure the server started correctly.")

print("\n📡 Detection service is now running!")
print("   Waiting for frames from your PC...")


## 📝 PC Configuration Summary

Once you have the ngrok public URL from above, configure your PC's `pc_frame_sender.py`:

```python
# In pc_frame_sender.py or when creating the PCFrameSender instance:
REMOTE_URL = "YOUR_NGROK_PUBLIC_URL_HERE"  # From ngrok output above
API_KEY = "YOUR_COLAB_API_KEY_HERE"          # The COLAB_API_KEY you set above

sender = PCFrameSender(
    remote_url=REMOTE_URL,
    api_key=API_KEY,
    camera_source="webcam",  # or "rtsp"
    webcam_index=0,
    frame_width=640,
    frame_height=640,
    fps=5,  # Start low and increase if needed
    show_preview=True
)
sender.run()
```

## 🔍 Troubleshooting

**If you see connection errors:**
1. Verify ngrok tunnel is running (check cell 4 output)
2. Verify your PC can reach the public URL (try in browser)
3. Check that the detection service health endpoint returns 200
4. Verify API keys match between PC and Colab
5. Check firewall settings on your PC

**If detection quality is poor:**
1. Increase FPS in pc_frame_sender.py (but don't exceed what your network/Colab can handle)
2. Ensure good lighting and frontal face poses
3. Verify similarity threshold matches between local and Colab (0.35)
4. Check that known faces are properly loaded in Colab (see gallery size in health check)

**If you want to use GPU:**
1. Change runtime type to GPU (Runtime → Change runtime type → Hardware accelerator → GPU)
2. Set USE_GPU = "true" in the configuration cell above
3. Restart the notebook and run all cells again

## 🛑 To Stop Everything

1. Interrupt the server cell (■) if running
2. Interrupt the ngrok tunnel cell (■) if running
3. Or simply restart the runtime (Runtime → Restart runtime)

---\n*Service ready. Awaiting frames from your PC.*